In [1]:
!pip -q install pymupdf pandas requests tqdm

In [6]:
!pip -q install transformers

In [2]:
import json
import pandas as pd
from pathlib import Path

# Load the frozen evaluation dataset
eval_path = Path("/content/eval_set.jsonl")

with open(eval_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

eval_df = pd.DataFrame(records)

# Identify unique financial documents
required_docs = eval_df[
    ["company", "doc_name", "doc_period"]
].drop_duplicates(subset=["doc_name"])

print("Total evaluation questions:", len(eval_df))
print("Unique source PDFs:", len(required_docs))

display(required_docs.head(10))

Total evaluation questions: 80
Unique source PDFs: 58


,company,doc_name,doc_period
0,Ulta Beauty,ULTABEAUTY_2023_10K,2023
1,American Water Works,AMERICANWATERWORKS_2020_10K,2020
2,Corning,CORNING_2020_10K,2020
3,American Express,AMERICANEXPRESS_2022_10K,2022
4,Walmart,WALMART_2018_10K,2018
5,Microsoft,MICROSOFT_2023_10K,2023
6,AMD,AMD_2015_10K,2015
7,Pfizer,Pfizer_2023Q2_10Q,2023
8,Block,BLOCK_2016_10K,2016
9,MGM Resorts,MGMRESORTS_2020_10K,2020


In [3]:

import requests
import time
from pathlib import Path
from tqdm.auto import tqdm

# Create a directory for PDFs
pdf_dir = Path("/content/data/pdfs")
pdf_dir.mkdir(parents=True, exist_ok=True)

# Official FinanceBench PDF source
base_url = (
    "https://raw.githubusercontent.com/"
    "patronus-ai/financebench/main/pdfs/"
)

download_results = []

for doc_name in tqdm(required_docs["doc_name"], desc="Downloading PDFs"):
    filename = f"{doc_name}.pdf"
    output_path = pdf_dir / filename
    url = base_url + filename

    try:
        # Skip if already downloaded
        if output_path.exists() and output_path.stat().st_size > 0:
            status = "Already downloaded"
        else:
            response = requests.get(url, timeout=90)
            response.raise_for_status()

            # Verify PDF signature before saving
            if not response.content.startswith(b"%PDF-"):
                raise ValueError("Downloaded content is not a PDF")

            output_path.write_bytes(response.content)
            status = "Downloaded"

        download_results.append({
            "doc_name": doc_name,
            "status": status
        })

    except Exception as e:
        download_results.append({
            "doc_name": doc_name,
            "status": f"Failed: {str(e)[:100]}"
        })

    time.sleep(0.1)

# Review results
download_df = pd.DataFrame(download_results)

print("\nDownload results:")
print(download_df["status"].value_counts())

print("\nPDF files saved:", len(list(pdf_dir.glob("*.pdf"))))

print("\nFailed documents:")
display(download_df[download_df["status"].str.startswith("Failed")])



Download results:
status
Downloaded    58
Name: count, dtype: int64

PDF files saved: 58

Failed documents:


,doc_name,status


In [4]:

import fitz
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Create a lookup of document metadata
doc_metadata = required_docs.set_index("doc_name").to_dict("index")

extracted_pages = []
extraction_errors = []

# Process all 58 financial documents
for pdf_path in tqdm(sorted(pdf_dir.glob("*.pdf")), desc="Extracting PDF text"):
    doc_name = pdf_path.stem

    try:
        metadata = doc_metadata[doc_name]

        with fitz.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf):
                text = page.get_text("text").strip()

                if text:
                    extracted_pages.append({
                        "company": metadata["company"],
                        "doc_name": doc_name,
                        "fiscal_year": int(metadata["doc_period"]),
                        "page_number": page_num,
                        "text": text
                    })

    except Exception as e:
        extraction_errors.append({
            "doc_name": doc_name,
            "error": str(e)
        })

# Convert results to DataFrame
pages_df = pd.DataFrame(extracted_pages)

print("Total documents processed:", len(list(pdf_dir.glob("*.pdf"))))
print("Pages with extracted text:", len(pages_df))
print("Documents with extracted text:", pages_df["doc_name"].nunique())
print("Extraction errors:", len(extraction_errors))

display(pages_df.head(3))


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Extracting PDF text:   0%|          | 0/58 [00:00<?, ?it/s]

Total documents processed: 58
Pages with extracted text: 8055
Documents with extracted text: 58
Extraction errors: 0


,company,doc_name,fiscal_year,page_number,text
0,3M,3M_2018_10K,2018,0,low\n \nUNITED STATES\nSECURITIES AND EXCHANGE...
1,3M,3M_2018_10K,2018,1,Table of Contents \n3M COMPANY \nFORM 10-K\nFo...
2,3M,3M_2018_10K,2018,2,Table of Contents \n \n \n \n \nBeginning\nPag...


In [5]:

# Collect ground-truth evidence pages
evidence_pages = set()

for record in records:
    for evidence in record["evidence"]:
        evidence_pages.add((
            evidence["doc_name"],
            evidence["page_number"]
        ))

# Collect extracted pages
extracted_page_keys = set(
    zip(pages_df["doc_name"], pages_df["page_number"])
)

# Identify missing evidence pages
missing_pages = evidence_pages - extracted_page_keys

print("Unique evidence pages:", len(evidence_pages))
print("Evidence pages found:", len(evidence_pages) - len(missing_pages))
print("Missing evidence pages:", len(missing_pages))

if missing_pages:
    print("\nMissing pages:")
    for doc_name, page_num in sorted(missing_pages):
        print(doc_name, "Page:", page_num)
else:
    print("\nAll evidence pages have extracted text!")


Unique evidence pages: 95
Evidence pages found: 95
Missing evidence pages: 0

All evidence pages have extracted text!


In [7]:

import json
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer

# Use the tokenizer associated with our chosen embedding model
tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)

CHUNK_SIZE = 384
CHUNK_OVERLAP = 64
STEP = CHUNK_SIZE - CHUNK_OVERLAP

chunks = []

for row in tqdm(
    pages_df.itertuples(index=False),
    total=len(pages_df),
    desc="Creating text chunks"
):
    token_ids = tokenizer.encode(
        row.text,
        add_special_tokens=False
    )

    if not token_ids:
        continue

    chunk_number = 0

    for start in range(0, len(token_ids), STEP):
        end = min(start + CHUNK_SIZE, len(token_ids))
        chunk_tokens = token_ids[start:end]

        if not chunk_tokens:
            continue

        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        ).strip()

        if not chunk_text:
            continue

        chunks.append({
            "chunk_id": (
                f"{row.doc_name}_p{row.page_number}_c{chunk_number}"
            ),
            "text": chunk_text,
            "company": row.company,
            "doc_name": row.doc_name,
            "fiscal_year": int(row.fiscal_year),
            "page_number": int(row.page_number)
        })

        chunk_number += 1

        if end == len(token_ids):
            break

# Save all chunks
output_dir = Path("/content/data")
output_dir.mkdir(parents=True, exist_ok=True)

chunks_path = output_dir / "chunks.jsonl"

with open(chunks_path, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("Total chunks created:", len(chunks))
print("Documents represented:", len(set(c["doc_name"] for c in chunks)))
print("File saved:", chunks_path)

print("\nExample chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:1200])


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Creating text chunks:   0%|          | 0/8055 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (903 > 512). Running this sequence through the model will result in indexing errors


Total chunks created: 19802
Documents represented: 58
File saved: /content/data/chunks.jsonl

Example chunk:
{
  "chunk_id": "3M_2018_10K_p0_c0",
  "text": "low united states securities and exchange commission washington, d. c. 20549 form 10 - k annual report pursuant to section 13 or 15 ( d ) of the securities exchange act of 1934 for the fiscal year ended december 31, 2018 commission file number 1 - 3285 3m company state of incorporation : delaware i. r. s. employer identification no. 41 - 0417775 principal executive offices : 3m center, st. paul, minnesota 55144 telephone number : ( 651 ) 733 - 1110 securities registered pursuant to section 12 ( b ) of the act : title of each class name of each exchange on which registered common stock, par value $. 01 per share 1. 500 % notes due 2026 floating rate notes due 2020 0. 375 % notes due 2022 0. 950 % notes due 2023 1. 750 % notes due 2030 1. 500 % notes due 2031 new york stock exchange, inc. chicago stock exchange, inc. new york stock e

In [8]:

import pandas as pd

chunks_df = pd.DataFrame(chunks)

# Check required metadata
required_columns = [
    "chunk_id",
    "text",
    "company",
    "doc_name",
    "fiscal_year",
    "page_number"
]

print("Total chunks:", len(chunks_df))
print("Unique chunk IDs:", chunks_df["chunk_id"].nunique())
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())

print("\nMissing values:")
print(chunks_df[required_columns].isna().sum())

# Calculate token lengths using the same tokenizer
token_lengths = [
    len(tokenizer.encode(text, add_special_tokens=False))
    for text in chunks_df["text"]
]

print("\nToken length statistics:")
print(pd.Series(token_lengths).describe())

print("\nChunks exceeding 384 tokens:",
      sum(length > 384 for length in token_lengths))

print("\nDocuments represented:",
      chunks_df["doc_name"].nunique())

# Ensure all evidence pages have at least one chunk
chunk_page_keys = set(
    zip(chunks_df["doc_name"], chunks_df["page_number"])
)

missing_evidence_pages = evidence_pages - chunk_page_keys

print("Evidence pages without chunks:",
      len(missing_evidence_pages))


Total chunks: 19802
Unique chunk IDs: 19802
Duplicate IDs: 0

Missing values:
chunk_id       0
text           0
company        0
doc_name       0
fiscal_year    0
page_number    0
dtype: int64

Token length statistics:
count    19802.000000
mean       318.581608
std        100.174760
min          1.000000
25%        263.000000
50%        384.000000
75%        384.000000
max        388.000000
dtype: float64

Chunks exceeding 384 tokens: 417

Documents represented: 58
Evidence pages without chunks: 0


In [9]:

import json
from pathlib import Path
from tqdm.auto import tqdm

MAX_TOKENS = 384
fixed_chunks = []

for chunk in tqdm(chunks, desc="Validating chunk lengths"):
    updated = chunk.copy()

    token_ids = tokenizer.encode(
        updated["text"],
        add_special_tokens=False,
        verbose=False
    )

    if len(token_ids) > MAX_TOKENS:
        # Reduce the text until re-tokenization is within limit
        while len(token_ids) > MAX_TOKENS:
            token_ids = token_ids[:-8]

            updated["text"] = tokenizer.decode(
                token_ids,
                skip_special_tokens=True
            ).strip()

            token_ids = tokenizer.encode(
                updated["text"],
                add_special_tokens=False,
                verbose=False
            )

    fixed_chunks.append(updated)

# Replace original chunks
chunks = fixed_chunks

# Save corrected chunks
chunks_path = Path("/content/data/chunks.jsonl")

with open(chunks_path, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

# Verify token lengths
new_lengths = [
    len(tokenizer.encode(
        chunk["text"],
        add_special_tokens=False,
        verbose=False
    ))
    for chunk in chunks
]

print("Total chunks:", len(chunks))
print("Maximum token length:", max(new_lengths))
print("Chunks exceeding 384:", sum(n > 384 for n in new_lengths))
print("Documents represented:", len(set(c["doc_name"] for c in chunks)))
print("Corrected file saved:", chunks_path)


Validating chunk lengths:   0%|          | 0/19802 [00:00<?, ?it/s]

Total chunks: 19802
Maximum token length: 384
Chunks exceeding 384: 0
Documents represented: 58
Corrected file saved: /content/data/chunks.jsonl


In [10]:
from pathlib import Path

chunks_path = Path("/content/data/chunks.jsonl")

size_mb = chunks_path.stat().st_size / (1024 * 1024)

print(f"File size: {size_mb:.2f} MB")
print("Total chunks:", len(chunks))

File size: 32.24 MB
Total chunks: 19802


In [11]:
from google.colab import files

files.download("/content/data/chunks.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>